# Machine Learning Assignment 2

**Dataset:** Breast Cancer Wisconsin (Diagnostic)  
**Goal:** Train and compare the five classifiers explicitly named in the assignment and save reusable model files for Streamlit.

In [1]:
import json
from pathlib import Path
import joblib
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef, confusion_matrix, classification_report

RANDOM_STATE = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print("Project root:", ROOT)
print("Imports successful")

Project root: /mnt/data/ml_assignment_2_breast_cancer
Imports successful


## 1. Dataset choice and preparation

The UCI Breast Cancer Wisconsin (Diagnostic) dataset has 569 instances and 30 numeric features. The scikit-learn packaged copy is used for reproducibility. The target is remapped so `1 = malignant` and `0 = benign`.

In [2]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
df["diagnosis"] = (df["target"] == 0).astype(int)
df = df.drop(columns=["target"])

feature_columns = [c for c in df.columns if c != "diagnosis"]
X = df[feature_columns]
y = df["diagnosis"]

print("Dataset shape:", df.shape)
print("Number of features:", len(feature_columns))
print("Target counts:")
print(y.value_counts().sort_index())
df.head()

Dataset shape: (569, 31)
Number of features: 30
Target counts:
diagnosis
0    357
1    212
Name: count, dtype: int64


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,diagnosis
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,1
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,1
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,1
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,1
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,1


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

test_data = X_test.copy()
test_data["diagnosis"] = y_test.values
test_data.to_csv(ROOT / "test_data.csv", index=False)
print("Saved test_data.csv")

Training rows: 455
Test rows: 114
Saved test_data.csv


## 2. Build the classifiers

In [4]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, min_samples_leaf=4, random_state=RANDOM_STATE),
    "kNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7)),
    ]),
    "Naive Bayes": Pipeline([
        ("scaler", StandardScaler()),
        ("model", GaussianNB()),
    ]),
    "Random Forest (Ensemble)": RandomForestClassifier(
        n_estimators=400, max_features="sqrt", random_state=RANDOM_STATE, n_jobs=-1
    ),
}
models

{'Logistic Regression': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model', LogisticRegression(max_iter=5000, random_state=42))]),
 'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=4, random_state=42),
 'kNN': Pipeline(steps=[('scaler', StandardScaler()),
                 ('model', KNeighborsClassifier(n_neighbors=7))]),
 'Naive Bayes': Pipeline(steps=[('scaler', StandardScaler()), ('model', GaussianNB())]),
 'Random Forest (Ensemble)': RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=42)}

## 3. Train and evaluate on the same held-out test data

In [5]:
rows = []
trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    rows.append({
        "ML Model Name": name,
        "Accuracy": accuracy_score(y_test, pred),
        "AUC": roc_auc_score(y_test, prob),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, pred),
    })
    trained_models[name] = model

metrics_df = pd.DataFrame(rows).sort_values(["MCC", "F1", "AUC"], ascending=False).reset_index(drop=True)
metrics_df

,ML Model Name,Accuracy,AUC,Precision,Recall,F1,MCC
0,Random Forest (Ensemble),0.973684,0.994213,1.000000,0.928571,0.962963,0.944155
1,Logistic Regression,0.964912,0.996032,0.975000,0.928571,0.951220,0.924518
2,kNN,0.956140,0.982474,0.974359,0.904762,0.938272,0.905824
3,Naive Bayes,0.921053,0.989087,0.923077,0.857143,0.888889,0.829162
4,Decision Tree,0.877193,0.965443,0.911765,0.738095,0.815789,0.734316


## 4. Detailed evaluation for the overall winner

In [6]:
winner_name = metrics_df.iloc[0]["ML Model Name"]
winner = trained_models[winner_name]
winner_pred = winner.predict(X_test)
print("Overall winner:", winner_name)
print("Confusion matrix:")
print(confusion_matrix(y_test, winner_pred, labels=[0, 1]))
print("\nClassification report:")
print(classification_report(y_test, winner_pred, target_names=["Benign", "Malignant"], zero_division=0))

Overall winner: Random Forest (Ensemble)
Confusion matrix:
[[72  0]
 [ 3 39]]

Classification report:
              precision    recall  f1-score   support

      Benign       0.96      1.00      0.98        72
   Malignant       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114



## 5. Save the trained models and results

In [7]:
model_dir = ROOT / "model"
results_dir = ROOT / "results"
model_dir.mkdir(exist_ok=True)
results_dir.mkdir(exist_ok=True)

model_files = {
    "Logistic Regression": "logistic_regression.joblib",
    "Decision Tree": "decision_tree.joblib",
    "kNN": "knn.joblib",
    "Naive Bayes": "naive_bayes.joblib",
    "Random Forest (Ensemble)": "random_forest.joblib",
}

for name, model in trained_models.items():
    joblib.dump(model, model_dir / model_files[name])
metrics_df.to_csv(results_dir / "model_metrics.csv", index=False)
print("Saved five models and results/model_metrics.csv")

Saved five models and results/model_metrics.csv


## 6. Observations

- **Logistic Regression:** excellent linear baseline after scaling, with very high AUC and strong MCC.
- **Decision Tree:** weakest held-out performance of the five; a single tree is more sensitive to the specific training sample.
- **kNN:** strong results after standardization; slightly below the best ensemble.
- **Naive Bayes:** good AUC, but the independence assumption is restrictive for correlated medical measurements.
- **Random Forest:** best overall result on this test split by MCC and F1 while retaining excellent AUC.

## 7. BITS Virtual Lab evidence

Run **all cells** in the BITS Virtual Lab. Take one genuine screenshot showing successful execution (for example, the metrics table plus the Virtual Lab browser/desktop context) and place that screenshot into your final submission PDF.